# M13 — Validation-Only Calibration and Class-Specialist Ensemble

**EEEM068 Applied Machine Learning — Calibration and Ensemble Experiment (no new training)**

**Main objective:** Can calibrated combinations of complementary models improve validation
QWK, macro-F1, class-specific recognition, and confidence reliability beyond the strongest
single model, without using the internal test set for any decision?

**M13 is an ensemble and calibration experiment, not another fine-grained ablation.** It does
not train any neural network. It loads validation-set logits from already-trained, frozen
checkpoints, calibrates each model's confidence via temperature scaling, and combines models
through several ensemble strategies — evaluated honestly via five-fold out-of-fold splitting,
not by fitting and testing on the identical rows.

**M13 never loads the internal test set.** The frozen, selected M13 system will be evaluated
against the test set exactly once, later, in a separate notebook (A01) — not here.

**Candidate models**: EfficientNet-B4 (selected baseline), DeiT-III (selected previous-exam
model), M07 (MaxViT reference), M11 (global-local fusion), M12 (combined fine-grained
guidance — included because it is currently the strongest MaxViT model). M09 and M10 are
included only if their validation logits can be reliably regenerated from their frozen
checkpoints; none of the five architecturally-diverse models above are retrained here.

**Important honesty note on obtaining logits (read before running):** M09-M12 saved raw
validation logits by design (`validation_logits.npy` or `logit_grade_*` prediction-CSV
columns). **M07, EfficientNet-B4, and DeiT-III were built in earlier notebook series that did
not save raw pre-softmax logits** — only class probabilities and/or predicted labels. For
those models, Section 5's loader function attempts, in order: (a) a saved `.npy` logit file,
(b) `logit_grade_*` columns in a saved predictions CSV, (c) as a **documented, flagged
fallback**, reconstructing an approximation via `log(probability + epsilon)` from saved
probabilities — this is **not** genuinely equivalent to real pre-softmax logits and is marked
as such everywhere it is used, since temperature scaling calibrated against a log-probability
reconstruction is measurably less trustworthy than calibration against true logits. If none of
these are available for a given model, the loader raises a clear, actionable error rather than
silently fabricating data — you will need to run a validation-only inference pass for that
specific model's frozen checkpoint (using that model's own preprocessing/architecture code
from its own notebook) and save its output in the expected format before M13 can include it.

| Setting | M13 value |
|---|---|
| Trains a new network | No |
| Uses internal test set | Never |
| Calibration method | Temperature scaling (per model) |
| Honest-evaluation protocol | 5-fold stratified out-of-fold |
| Fold seed | 42 |
| Ensemble variants evaluated | A: uncalibrated equal average, B: calibrated equal average, C: calibrated global-weighted, D: calibrated class-specialist |
| Final selection rule | OOF QWK -> macro-F1 -> NLL -> ECE -> simpler ensemble |
| High-confidence error threshold | 0.80 |

## 2. Imports and reproducibility

In [ ]:
import json
import random
import warnings
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    average_precision_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold

print("Imports OK.")

In [ ]:
SEED = 42
FOLD_SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Seed: {SEED}  Fold seed: {FOLD_SEED}")

## 3. Paths and candidate-model configuration

Every candidate model's expected log directory. Paths for EfficientNet-B4 and DeiT-III are
best-effort based on this project's naming conventions from their own notebook series — if
they do not match your actual saved run, update `CANDIDATE_MODELS` below before running
Section 5.

In [ ]:
EXPERIMENT_ID = "M13"
RUN_NAME = "M13_calibrated_class_specialist_ensemble"

PROJECT_ROOT = Path(
    "/scratch/New AML/EEEM068-LSA-Diabetic-Retinopathy"
)

LOG_DIR = PROJECT_ROOT / "logs" / "maxvit_tiny" / RUN_NAME
FIGURE_DIR = PROJECT_ROOT / "results" / "figures" / "maxvit_tiny" / RUN_NAME
for directory in [LOG_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

NUM_CLASSES = 5
CLASS_NAMES = ["No DR", "Mild", "Moderate", "Severe", "Proliferative DR"]
HIGH_CONFIDENCE_THRESHOLD = 0.80
N_OOF_FOLDS = 5
WEIGHT_GRID = [0.0, 0.25, 0.5, 0.75, 1.0]  # M13-D's coarse per-class weight search

# --- Candidate models. "required" models must load successfully; "optional" models are
# skipped with a clear message if their logits cannot be obtained (M09/M10 per the spec). ---
CANDIDATE_MODELS = {
    "EfficientNet-B4": {
        "log_dir": PROJECT_ROOT / "logs" / "efficientnet_b4" / "exp02_p1_clipped_weighted_ce_earlystop_seed42",
        "required": True,
    },
    "DeiT-III": {
        "log_dir": PROJECT_ROOT / "logs" / "deit3_b16" / "D07_two_phase_finetuning",
        "required": True,
    },
    "M07": {
        "log_dir": PROJECT_ROOT / "logs" / "maxvit_tiny" / "M07_ordinal_aware_loss",
        "required": True,
    },
    "M09": {
        "log_dir": PROJECT_ROOT / "logs" / "maxvit_tiny" / "M09_detection_guided_regions",
        "required": False,
    },
    "M10": {
        "log_dir": PROJECT_ROOT / "logs" / "maxvit_tiny" / "M10_pseudo_segmentation_guidance",
        "required": False,
    },
    "M11": {
        "log_dir": PROJECT_ROOT / "logs" / "maxvit_tiny" / "M11_global_local_crop_fusion",
        "required": True,
    },
    "M12": {
        "log_dir": PROJECT_ROOT / "logs" / "maxvit_tiny" / "M12_combined_fine_grained_guidance",
        "required": True,
    },
}

print(f"Experiment : {EXPERIMENT_ID} / {RUN_NAME}")
print(f"Logs -> {LOG_DIR}")
print("Candidate models configured:", list(CANDIDATE_MODELS.keys()))

## 4. Validation-only and no-test guards

In [ ]:
loaded_splits = {"validation"}
assert "test" not in loaded_splits

print("M13 uses validation data only.")
print("Internal test set loaded: False.")

## 5. Load validation logits and metadata

`get_model_logits()` tries, in order: (1) a saved `.npy` logit file, (2) `logit_grade_*`
columns in a saved predictions CSV, (3) a flagged, approximate reconstruction from saved
probabilities via `log(prob + eps)`. Every model's `logit_source` is recorded so calibration
results can be interpreted with the right amount of trust — genuine logits vs. a
probability-derived approximation are not equally reliable inputs to temperature scaling.

In [ ]:
LOGIT_EPSILON = 1e-8


def _find_image_id_column(df: pd.DataFrame) -> str:
    for candidate in ["image_id", "image"]:
        if candidate in df.columns:
            return candidate
    raise ValueError(f"No image identifier column found among: {list(df.columns)}")


def _find_true_label_column(df: pd.DataFrame) -> str:
    for candidate in ["true_label", "true_grade"]:
        if candidate in df.columns:
            return candidate
    raise ValueError(f"No true-label column found among: {list(df.columns)}")


def get_model_logits(model_name: str, log_dir: Path) -> dict:
    """Returns {'logits': (N,5) array, 'labels': (N,) array, 'image_ids': list[str],
    'logit_source': 'raw_npy' | 'raw_csv' | 'reconstructed_from_probabilities'}."""

    npy_logits_path = log_dir / "validation_logits.npy"
    npy_labels_path = log_dir / "validation_labels.npy"
    npy_ids_path = log_dir / "validation_image_ids.csv"
    if npy_logits_path.exists() and npy_labels_path.exists() and npy_ids_path.exists():
        logits = np.load(npy_logits_path)
        labels = np.load(npy_labels_path)
        image_ids = pd.read_csv(npy_ids_path)["image_id"].astype(str).tolist()
        return {"logits": logits, "labels": labels, "image_ids": image_ids, "logit_source": "raw_npy"}

    pred_csv_path = log_dir / "validation_predictions.csv"
    if pred_csv_path.exists():
        df = pd.read_csv(pred_csv_path)
        id_col = _find_image_id_column(df)
        label_col = _find_true_label_column(df)

        logit_cols = [f"logit_grade_{c}" for c in range(NUM_CLASSES)]
        if all(col in df.columns for col in logit_cols):
            logits = df[logit_cols].to_numpy(dtype=np.float64)
            return {
                "logits": logits, "labels": df[label_col].to_numpy(),
                "image_ids": df[id_col].astype(str).tolist(), "logit_source": "raw_csv",
            }

        prob_cols = [f"prob_grade_{c}" for c in range(NUM_CLASSES)]
        if not all(col in df.columns for col in prob_cols):
            prob_cols = [f"prob_{c}" for c in range(NUM_CLASSES)]
        if all(col in df.columns for col in prob_cols):
            probs = df[prob_cols].to_numpy(dtype=np.float64)
            approx_logits = np.log(probs + LOGIT_EPSILON)
            return {
                "logits": approx_logits, "labels": df[label_col].to_numpy(),
                "image_ids": df[id_col].astype(str).tolist(),
                "logit_source": "reconstructed_from_probabilities",
            }

    raise FileNotFoundError(
        f"Could not obtain validation logits for '{model_name}' from {log_dir}. "
        "Run a validation-only inference pass for this model's frozen checkpoint (using its "
        "own architecture/preprocessing code) and save validation_logits.npy / "
        "validation_labels.npy / validation_image_ids.csv (or a validation_predictions.csv "
        "with logit_grade_0..4 or prob_grade_0..4 columns) before including it in M13."
    )


loaded_model_data = {}
for model_name, model_config in CANDIDATE_MODELS.items():
    try:
        loaded_model_data[model_name] = get_model_logits(model_name, model_config["log_dir"])
        source = loaded_model_data[model_name]["logit_source"]
        flag = "" if source != "reconstructed_from_probabilities" else "  [APPROXIMATE - see Section 1 note]"
        print(f"{model_name:<16s}: {len(loaded_model_data[model_name]['image_ids']):,} rows, source={source}{flag}")
    except FileNotFoundError as error:
        if model_config["required"]:
            raise
        print(f"{model_name:<16s}: SKIPPED (optional) - {error}")

CANDIDATE_MODEL_NAMES = list(loaded_model_data.keys())
print(f"\nModels included in M13: {CANDIDATE_MODEL_NAMES}")

## 6. Align models by image_id

Every model must cover exactly the same validation image set. Joining is done by
`image_id`, never by row position alone — different notebooks' DataLoaders may have iterated
their validation sets in different orders.

In [ ]:
id_sets = {name: set(data["image_ids"]) for name, data in loaded_model_data.items()}
common_ids = set.intersection(*id_sets.values())
union_ids = set.union(*id_sets.values())

if common_ids != union_ids:
    missing_report = {
        name: sorted(union_ids - ids)[:5] for name, ids in id_sets.items() if ids != union_ids
    }
    print("WARNING: not all models share the identical validation image set.")
    for name, missing_examples in missing_report.items():
        print(f"  {name} is missing (example ids): {missing_examples}")
    print(f"Proceeding with the {len(common_ids):,} images common to every included model.")
else:
    print("All included models cover the identical validation image set.")

aligned_ids = sorted(common_ids)
assert len(set(aligned_ids)) == len(aligned_ids)  # all_model_ids_are_unique, post-dedup

aligned_labels = None
aligned_logits = {}
for model_name, data in loaded_model_data.items():
    id_to_row = {image_id: row_idx for row_idx, image_id in enumerate(data["image_ids"])}
    row_order = [id_to_row[image_id] for image_id in aligned_ids]
    aligned_logits[model_name] = data["logits"][row_order]
    model_labels = np.asarray(data["labels"])[row_order]
    if aligned_labels is None:
        aligned_labels = model_labels
    else:
        assert np.array_equal(aligned_labels, model_labels), (
            f"{model_name}'s labels do not match after alignment - a genuine data-integrity problem."
        )

print(f"Validation samples aligned: {len(aligned_ids)}")
print("Internal test set loaded: False")

aligned_model_manifest = pd.DataFrame({
    "model": CANDIDATE_MODEL_NAMES,
    "logit_source": [loaded_model_data[name]["logit_source"] for name in CANDIDATE_MODEL_NAMES],
    "n_aligned_rows": [len(aligned_ids)] * len(CANDIDATE_MODEL_NAMES),
})
aligned_model_manifest.to_csv(LOG_DIR / "aligned_model_manifest.csv", index=False)
print(f"Saved -> {LOG_DIR / 'aligned_model_manifest.csv'}")

## 7. Validate labels, dimensions and finite values

In [ ]:
assert len(aligned_ids) == len(set(aligned_ids))  # all_model_ids_are_unique
for model_name in CANDIDATE_MODEL_NAMES:
    assert aligned_logits[model_name].shape[0] == len(aligned_ids)  # all_models_have_same_image_id_set
    assert np.array_equal(
        np.asarray(loaded_model_data[model_name]["labels"])[
            [ {img: i for i, img in enumerate(loaded_model_data[model_name]["image_ids"])}[a] for a in aligned_ids ]
        ],
        aligned_labels,
    )  # all_models_have_same_labels
    assert aligned_logits[model_name].shape[1] == NUM_CLASSES  # number_of_classes == 5
    assert np.isfinite(aligned_logits[model_name]).all(), f"{model_name} has non-finite logits."  # all_logits_are_finite

assert "test" not in loaded_splits  # internal_test_loaded is False

print("All alignment/validation assertions PASSED.")
print(f"Validation samples aligned: {len(aligned_ids)}")
print("Internal test set loaded: False")

## 8. Single-model uncalibrated metrics

Baseline numbers before any calibration or ensembling — every later comparison is against
this.

In [ ]:
def logits_to_probs(logits: np.ndarray) -> np.ndarray:
    logits_t = torch.tensor(logits, dtype=torch.float64)
    return F.softmax(logits_t, dim=1).numpy()


def compute_core_metrics(labels: np.ndarray, preds: np.ndarray) -> dict:
    absolute_error = np.abs(preds - labels)
    return {
        "qwk": cohen_kappa_score(labels, preds, weights="quadratic"),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(labels, preds),
        "accuracy": accuracy_score(labels, preds),
        "mae": float(absolute_error.mean()),
        "within_one_grade_accuracy": float((absolute_error <= 1).mean()),
        "large_grade_error_rate": float((absolute_error > 1).mean()),
    }


single_model_uncalibrated_rows = []
for model_name in CANDIDATE_MODEL_NAMES:
    probs = logits_to_probs(aligned_logits[model_name])
    preds = probs.argmax(axis=1)
    metrics = compute_core_metrics(aligned_labels, preds)
    metrics["model"] = model_name
    metrics["logit_source"] = loaded_model_data[model_name]["logit_source"]
    single_model_uncalibrated_rows.append(metrics)

single_model_uncalibrated_df = pd.DataFrame(single_model_uncalibrated_rows).set_index("model")
print(single_model_uncalibrated_df[["qwk", "macro_f1", "balanced_accuracy", "accuracy", "mae", "logit_source"]])

## 9. Calibration metric functions

NLL, expected calibration error (ECE, 15 equal-width confidence bins by default), and Brier
score.

In [ ]:
def negative_log_likelihood(labels: np.ndarray, probs: np.ndarray) -> float:
    return float(log_loss(labels, probs, labels=list(range(NUM_CLASSES))))


def expected_calibration_error(labels: np.ndarray, probs: np.ndarray, n_bins: int = 15) -> dict:
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    correctness = (predictions == labels).astype(float)

    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_records = []
    ece = 0.0
    for bin_idx in range(n_bins):
        lo, hi = bin_edges[bin_idx], bin_edges[bin_idx + 1]
        in_bin = (confidences > lo) & (confidences <= hi) if bin_idx > 0 else (confidences >= lo) & (confidences <= hi)
        count = int(in_bin.sum())
        if count == 0:
            bin_records.append({"bin_lo": lo, "bin_hi": hi, "count": 0, "mean_confidence": np.nan, "accuracy": np.nan, "gap": np.nan})
            continue
        mean_conf = float(confidences[in_bin].mean())
        acc = float(correctness[in_bin].mean())
        gap = abs(mean_conf - acc)
        ece += (count / len(confidences)) * gap
        bin_records.append({"bin_lo": lo, "bin_hi": hi, "count": count, "mean_confidence": mean_conf, "accuracy": acc, "gap": gap})

    return {"ece": float(ece), "bins": pd.DataFrame(bin_records)}


def brier_score(labels: np.ndarray, probs: np.ndarray) -> float:
    one_hot = np.eye(NUM_CLASSES)[labels]
    return float(np.mean(np.sum((probs - one_hot) ** 2, axis=1)))


print("Calibration metric functions ready (NLL, 15-bin ECE, Brier).")

## 10. Temperature-scaling implementation

`temperature = softplus(raw_temperature) + 1e-6`, optimised by minimising multiclass NLL on
the given (fold-restricted, during OOF) logits via gradient descent on the single scalar
parameter — a bounded, positive-temperature optimiser as required, not an unconstrained
per-class transform. Temperature scaling changes only the sharpness of the probability
distribution — since `argmax(logits / T) == argmax(logits)` for any `T > 0`, the predicted
class (and therefore QWK/accuracy/F1) is mathematically unchanged by this step; only
confidence (and therefore NLL/ECE/Brier) changes. This is stated explicitly rather than left
implicit, since it is easy to mistake calibration for a performance improvement.

In [ ]:
def fit_temperature(logits: np.ndarray, labels: np.ndarray, max_iter: int = 200, lr: float = 0.05) -> float:
    logits_t = torch.tensor(logits, dtype=torch.float64)
    labels_t = torch.tensor(labels, dtype=torch.long)

    raw_temperature = torch.zeros(1, dtype=torch.float64, requires_grad=True)
    optimizer = torch.optim.LBFGS([raw_temperature], lr=lr, max_iter=max_iter)

    def closure():
        optimizer.zero_grad()
        temperature = F.softplus(raw_temperature) + 1e-6
        calibrated_logits = logits_t / temperature
        loss = F.cross_entropy(calibrated_logits, labels_t)
        loss.backward()
        return loss

    optimizer.step(closure)

    with torch.no_grad():
        final_temperature = float((F.softplus(raw_temperature) + 1e-6).item())
    return final_temperature


def apply_temperature(logits: np.ndarray, temperature: float) -> np.ndarray:
    return logits_to_probs(logits / temperature)


print("Temperature-scaling implementation ready (softplus-parameterised, NLL-minimising).")

## 11. Five-fold out-of-fold framework

Because the same validation set is used for both calibration and ensemble-design decisions,
every headline comparison in this notebook is made **out-of-fold**: temperatures and ensemble
weights are fit on 4 folds and applied only to the 5th, held-out fold, for all 5 folds in
turn, then concatenated. This avoids the optimistic bias of fitting and evaluating on
identical rows. A fixed fold seed (42) makes the split reproducible.

In [ ]:
skf = StratifiedKFold(n_splits=N_OOF_FOLDS, shuffle=True, random_state=FOLD_SEED)
fold_assignment = np.zeros(len(aligned_labels), dtype=int)
for fold_idx, (_, held_out_idx) in enumerate(skf.split(np.zeros(len(aligned_labels)), aligned_labels)):
    fold_assignment[held_out_idx] = fold_idx

print(f"Stratified {N_OOF_FOLDS}-fold split ready (fold_seed={FOLD_SEED}).")
print("Fold sizes:", np.bincount(fold_assignment))
print("Class distribution preserved per fold (stratified):")
for fold_idx in range(N_OOF_FOLDS):
    print(f"  fold {fold_idx}: {np.bincount(aligned_labels[fold_assignment == fold_idx], minlength=NUM_CLASSES)}")

## 12. Calibrate individual models

Temperature fit on the **complete** validation set per model, for reporting purposes (matches
the spec's request for saved before/after NLL/ECE/Brier and QWK/macro-F1). As stated in
Section 10, QWK and macro-F1 are mathematically unchanged by temperature scaling — reported
here to make that explicit and verifiable, not because a change was expected.

In [ ]:
single_model_calibration_rows = []
fitted_temperatures = {}

for model_name in CANDIDATE_MODEL_NAMES:
    logits = aligned_logits[model_name]

    uncalibrated_probs = logits_to_probs(logits)
    uncalibrated_preds = uncalibrated_probs.argmax(axis=1)
    uncalibrated_core = compute_core_metrics(aligned_labels, uncalibrated_preds)

    temperature = fit_temperature(logits, aligned_labels)
    fitted_temperatures[model_name] = temperature
    calibrated_probs = apply_temperature(logits, temperature)
    calibrated_preds = calibrated_probs.argmax(axis=1)
    calibrated_core = compute_core_metrics(aligned_labels, calibrated_preds)

    assert np.array_equal(uncalibrated_preds, calibrated_preds), (
        "Temperature scaling changed predicted classes - this should be mathematically impossible for T>0."
    )

    single_model_calibration_rows.append({
        "model": model_name,
        "temperature": temperature,
        "uncalibrated_nll": negative_log_likelihood(aligned_labels, uncalibrated_probs),
        "calibrated_nll": negative_log_likelihood(aligned_labels, calibrated_probs),
        "uncalibrated_ece": expected_calibration_error(aligned_labels, uncalibrated_probs)["ece"],
        "calibrated_ece": expected_calibration_error(aligned_labels, calibrated_probs)["ece"],
        "uncalibrated_brier": brier_score(aligned_labels, uncalibrated_probs),
        "calibrated_brier": brier_score(aligned_labels, calibrated_probs),
        "uncalibrated_qwk": uncalibrated_core["qwk"], "calibrated_qwk": calibrated_core["qwk"],
        "uncalibrated_macro_f1": uncalibrated_core["macro_f1"], "calibrated_macro_f1": calibrated_core["macro_f1"],
    })

single_model_calibration_df = pd.DataFrame(single_model_calibration_rows)
print(single_model_calibration_df.to_string(index=False))

single_model_calibration_df.to_csv(LOG_DIR / "single_model_calibration_metrics.csv", index=False)
single_model_calibration_df[["model", "temperature", "uncalibrated_nll", "calibrated_nll"]].to_csv(
    LOG_DIR / "temperature_scaling_results.csv", index=False
)
print(f"Saved -> {LOG_DIR / 'single_model_calibration_metrics.csv'}")
print(f"Saved -> {LOG_DIR / 'temperature_scaling_results.csv'}")
print(
    "\nNote: QWK and macro-F1 are identical before/after calibration for every model above "
    "(verified by assertion) - temperature scaling changes confidence, not predicted class."
)

## 13. M13-A — Uncalibrated equal average

The simplest reference ensemble: mean of raw (uncalibrated) per-model probabilities.

In [ ]:
def ensemble_a_uncalibrated_equal_average(logits_by_model: dict, model_names: list) -> np.ndarray:
    probs_list = [logits_to_probs(logits_by_model[name]) for name in model_names]
    return np.mean(probs_list, axis=0)


print("M13-A defined: mean(model_probabilities), no calibration.")

## 14. M13-B — Calibrated equal average

Each model's own temperature is applied first, then probabilities are averaged equally. Tests
whether confidence calibration alone improves probability fusion, independent of any weight
optimisation.

In [ ]:
def ensemble_b_calibrated_equal_average(logits_by_model: dict, temperatures: dict, model_names: list) -> np.ndarray:
    probs_list = [apply_temperature(logits_by_model[name], temperatures[name]) for name in model_names]
    return np.mean(probs_list, axis=0)


print("M13-B defined: mean(calibrated model_probabilities).")

## 15. M13-C — Calibrated global-weighted ensemble

Non-negative weights (summing to 1) chosen from validation data via a coarse grid search —
not hundreds of unconstrained parameters. Objective: maximise QWK primarily, macro-F1 as
tie-break, minimise NLL as second tie-break.

In [ ]:
def _normalise_weights(raw_weights: np.ndarray) -> np.ndarray:
    total = raw_weights.sum()
    if total <= 0:
        return np.ones_like(raw_weights) / len(raw_weights)
    return raw_weights / total


def fit_global_weights(
    calibrated_probs_by_model: dict, labels: np.ndarray, model_names: list, weight_grid: list = WEIGHT_GRID,
) -> np.ndarray:
    """Coarse grid search over non-negative weights (normalised to sum to 1).
    Small, transparent search space - not an unconstrained optimiser."""
    best_weights = None
    best_score = (-np.inf, -np.inf, np.inf)  # (qwk, macro_f1, -nll)

    if len(model_names) <= 4:
        candidate_grids = [weight_grid] * len(model_names)
        from itertools import product as _product
        search_space = _product(*candidate_grids)
    else:
        # For more than 4 models, a full grid is too large; use a coarser 3-level grid.
        coarse_grid = [0.0, 0.5, 1.0]
        candidate_grids = [coarse_grid] * len(model_names)
        from itertools import product as _product
        search_space = _product(*candidate_grids)

    for raw_weights_tuple in search_space:
        raw_weights = np.array(raw_weights_tuple, dtype=float)
        if raw_weights.sum() <= 0:
            continue
        weights = _normalise_weights(raw_weights)

        combined = sum(w * calibrated_probs_by_model[name] for w, name in zip(weights, model_names))
        preds = combined.argmax(axis=1)
        qwk = cohen_kappa_score(labels, preds, weights="quadratic")
        macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
        nll = negative_log_likelihood(labels, combined)
        score = (qwk, macro_f1, -nll)
        if score > best_score:
            best_score = score
            best_weights = weights

    return best_weights


def ensemble_c_calibrated_weighted(
    logits_by_model: dict, temperatures: dict, weights: np.ndarray, model_names: list,
) -> np.ndarray:
    probs_list = [apply_temperature(logits_by_model[name], temperatures[name]) for name in model_names]
    return sum(w * p for w, p in zip(weights, probs_list))


print("M13-C defined: grid-searched non-negative global weights, sum to 1.")

## 16. M13-D — Calibrated class-specialist ensemble

Per-class weights, coarse-grid-searched from `{0.0, 0.25, 0.5, 0.75, 1.0}`, capped at **at
most 2 active (non-zero) models per class** to control complexity and avoid overfitting the
specialist ensemble to validation noise. All weights non-negative, normalised per class.

**Search objective, corrected to match how the final ensemble is actually assembled**: each
class's weight column is selected by evaluating that class's own **one-vs-rest** score
directly (lowest binary Brier loss, then highest one-vs-rest average precision, then fewer
active models as the final tie-break) — not by combining trial weights into a full 5-class
prediction using that one class's weights applied everywhere. The latter would search for
weights that make a *poor whole-vector* choice look good for one class in isolation, which is
not the system M13-D actually assembles (where each class column is independent). The
completed M13-D ensemble is still evaluated honestly afterward, out-of-fold, using QWK and
macro-F1 like every other variant — only the per-class *search* objective changed, not the
final evaluation.

In [ ]:
def fit_class_specialist_weights(
    calibrated_probs_by_model: dict,
    labels: np.ndarray,
    model_names: list,
    weight_grid: list = WEIGHT_GRID,
    max_active_models_per_class: int = 2,
) -> np.ndarray:
    """
    Fit non-negative weights independently for each class.

    Each candidate weight vector is evaluated only on that class's
    one-vs-rest probability score. This matches the later M13-D design,
    where each class receives its own model-weight column.

    Selection objective:
      1. Lowest one-vs-rest Brier loss
      2. Highest one-vs-rest average precision
      3. Fewer active models
    """
    from itertools import combinations as _combinations
    from itertools import product as _product

    num_models = len(model_names)
    class_weights = np.zeros(
        (num_models, NUM_CLASSES),
        dtype=np.float64,
    )

    for class_id in range(NUM_CLASSES):
        binary_labels = (
            labels == class_id
        ).astype(np.float64)

        best_weights = None
        best_key = None

        subsets = []

        for subset_size in range(
            1,
            min(max_active_models_per_class, num_models) + 1,
        ):
            subsets.extend(
                _combinations(
                    range(num_models),
                    subset_size,
                )
            )

        for subset in subsets:
            for raw_values in _product(
                weight_grid,
                repeat=len(subset),
            ):
                if sum(raw_values) <= 0:
                    continue

                trial_weights = np.zeros(
                    num_models,
                    dtype=np.float64,
                )

                for model_index, value in zip(
                    subset,
                    raw_values,
                ):
                    trial_weights[model_index] = value

                trial_weights = _normalise_weights(
                    trial_weights
                )

                class_score = np.zeros(
                    len(labels),
                    dtype=np.float64,
                )

                for model_index, model_name in enumerate(
                    model_names
                ):
                    class_score += (
                        trial_weights[model_index]
                        * calibrated_probs_by_model[
                            model_name
                        ][:, class_id]
                    )

                class_score = np.clip(
                    class_score,
                    1e-8,
                    1.0 - 1e-8,
                )

                binary_brier = float(
                    np.mean(
                        (
                            class_score
                            - binary_labels
                        ) ** 2
                    )
                )

                class_ap = float(
                    average_precision_score(
                        binary_labels,
                        class_score,
                    )
                )

                active_models = int(
                    np.sum(
                        trial_weights > 1e-8
                    )
                )

                # Lower tuple is better:
                # Brier first, then negative AP, then complexity.
                candidate_key = (
                    binary_brier,
                    -class_ap,
                    active_models,
                )

                if (
                    best_key is None
                    or candidate_key < best_key
                ):
                    best_key = candidate_key
                    best_weights = trial_weights.copy()

        if best_weights is None:
            best_weights = (
                np.ones(
                    num_models,
                    dtype=np.float64,
                )
                / num_models
            )

        class_weights[:, class_id] = best_weights

    assert np.isfinite(class_weights).all()
    assert np.all(class_weights >= 0)

    for class_id in range(NUM_CLASSES):
        assert np.isclose(
            class_weights[:, class_id].sum(),
            1.0,
        )

    return class_weights


def ensemble_d_class_specialist(
    logits_by_model: dict, temperatures: dict, class_weights: np.ndarray, model_names: list,
) -> np.ndarray:
    calibrated_probs = {name: apply_temperature(logits_by_model[name], temperatures[name]) for name in model_names}
    combined_score = np.column_stack([
        sum(class_weights[i, c] * calibrated_probs[model_names[i]][:, c] for i in range(len(model_names)))
        for c in range(NUM_CLASSES)
    ])
    row_sums = combined_score.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    return combined_score / row_sums


print("M13-D defined: coarse per-class weight grid, <=2 active models per class.")

## 17. Out-of-fold comparison

For each of the 5 folds: fit temperatures (Section 10) and, where applicable, ensemble
weights (Sections 15-16) on the other 4 folds only, then apply everything to the held-out
fold. Concatenating all 5 held-out predictions gives an honest, non-leaked estimate for every
variant — this is the basis for model selection in Section 19, not the full-validation refit.

In [ ]:
oof_probs = {"A": np.zeros((len(aligned_labels), NUM_CLASSES)), "B": np.zeros((len(aligned_labels), NUM_CLASSES)),
             "C": np.zeros((len(aligned_labels), NUM_CLASSES)), "D": np.zeros((len(aligned_labels), NUM_CLASSES))}
oof_fold_diagnostics = []

for fold_idx in range(N_OOF_FOLDS):
    train_mask = fold_assignment != fold_idx
    held_out_mask = fold_assignment == fold_idx

    train_logits = {name: aligned_logits[name][train_mask] for name in CANDIDATE_MODEL_NAMES}
    train_labels = aligned_labels[train_mask]
    held_out_logits = {name: aligned_logits[name][held_out_mask] for name in CANDIDATE_MODEL_NAMES}

    fold_temperatures = {name: fit_temperature(train_logits[name], train_labels) for name in CANDIDATE_MODEL_NAMES}

    train_calibrated_probs = {
        name: apply_temperature(train_logits[name], fold_temperatures[name]) for name in CANDIDATE_MODEL_NAMES
    }
    fold_global_weights = fit_global_weights(train_calibrated_probs, train_labels, CANDIDATE_MODEL_NAMES)
    fold_class_weights = fit_class_specialist_weights(train_calibrated_probs, train_labels, CANDIDATE_MODEL_NAMES)

    oof_probs["A"][held_out_mask] = ensemble_a_uncalibrated_equal_average(held_out_logits, CANDIDATE_MODEL_NAMES)
    oof_probs["B"][held_out_mask] = ensemble_b_calibrated_equal_average(held_out_logits, fold_temperatures, CANDIDATE_MODEL_NAMES)
    oof_probs["C"][held_out_mask] = ensemble_c_calibrated_weighted(held_out_logits, fold_temperatures, fold_global_weights, CANDIDATE_MODEL_NAMES)
    oof_probs["D"][held_out_mask] = ensemble_d_class_specialist(held_out_logits, fold_temperatures, fold_class_weights, CANDIDATE_MODEL_NAMES)

    oof_fold_diagnostics.append({
        "fold": fold_idx, "n_held_out": int(held_out_mask.sum()),
        **{f"temperature_{name}": fold_temperatures[name] for name in CANDIDATE_MODEL_NAMES},
        **{f"global_weight_{name}": fold_global_weights[i] for i, name in enumerate(CANDIDATE_MODEL_NAMES)},
    })
    print(f"Fold {fold_idx}: temperatures and weights fit on training folds, applied to {int(held_out_mask.sum())} held-out rows.")

oof_fold_diagnostics_df = pd.DataFrame(oof_fold_diagnostics)
print("\nPer-fold diagnostics (temperatures/global weights vary slightly fold to fold, as expected):")
print(oof_fold_diagnostics_df.round(4).to_string(index=False))

oof_fold_diagnostics_df.to_csv(LOG_DIR / "oof_fold_diagnostics.csv", index=False)
print(f"Saved -> {LOG_DIR / 'oof_fold_diagnostics.csv'}")

In [ ]:
oof_variant_metrics = {}
for variant in ["A", "B", "C", "D"]:
    preds = oof_probs[variant].argmax(axis=1)
    core = compute_core_metrics(aligned_labels, preds)
    oof_variant_metrics[variant] = {
        **core,
        "nll": negative_log_likelihood(aligned_labels, oof_probs[variant]),
        "ece": expected_calibration_error(aligned_labels, oof_probs[variant])["ece"],
        "brier": brier_score(aligned_labels, oof_probs[variant]),
    }

oof_comparison_df = pd.DataFrame(oof_variant_metrics).T
oof_comparison_df.index.name = "variant"
print("\n=== Out-of-fold ensemble comparison (the honest numbers) ===")
print(oof_comparison_df.round(4).to_string())

oof_comparison_df.reset_index().to_csv(LOG_DIR / "oof_ensemble_metrics.csv", index=False)
print(f"\nSaved -> {LOG_DIR / 'oof_ensemble_metrics.csv'}")

## 18. Complexity and stability checks

Compares each variant's **in-sample** (full-validation-fit, applied back to the same rows —
optimistic by construction) performance against its **out-of-fold** performance. A large gap
signals overfitting to validation noise, which is exactly the failure mode Section 15's
selection rule (Section 19) is designed to avoid rewarding.

In [ ]:
full_fit_calibrated_probs = {
    name: apply_temperature(aligned_logits[name], fitted_temperatures[name]) for name in CANDIDATE_MODEL_NAMES
}
full_fit_global_weights = fit_global_weights(full_fit_calibrated_probs, aligned_labels, CANDIDATE_MODEL_NAMES)
full_fit_class_weights = fit_class_specialist_weights(full_fit_calibrated_probs, aligned_labels, CANDIDATE_MODEL_NAMES)

in_sample_probs = {
    "A": ensemble_a_uncalibrated_equal_average(aligned_logits, CANDIDATE_MODEL_NAMES),
    "B": ensemble_b_calibrated_equal_average(aligned_logits, fitted_temperatures, CANDIDATE_MODEL_NAMES),
    "C": ensemble_c_calibrated_weighted(aligned_logits, fitted_temperatures, full_fit_global_weights, CANDIDATE_MODEL_NAMES),
    "D": ensemble_d_class_specialist(aligned_logits, fitted_temperatures, full_fit_class_weights, CANDIDATE_MODEL_NAMES),
}

stability_rows = []
for variant in ["A", "B", "C", "D"]:
    in_sample_preds = in_sample_probs[variant].argmax(axis=1)
    in_sample_qwk = cohen_kappa_score(aligned_labels, in_sample_preds, weights="quadratic")
    oof_qwk = oof_variant_metrics[variant]["qwk"]
    stability_rows.append({
        "variant": variant, "in_sample_qwk": in_sample_qwk, "oof_qwk": oof_qwk,
        "optimism_gap": in_sample_qwk - oof_qwk,
    })

stability_df = pd.DataFrame(stability_rows)
print(stability_df.round(4).to_string(index=False))

stability_df.to_csv(LOG_DIR / "ensemble_stability.csv", index=False)
print(f"Saved -> {LOG_DIR / 'ensemble_stability.csv'}")

active_models_per_class = (full_fit_class_weights > 1e-6).sum(axis=0)
print("\nActive (non-zero-weight) models per class in M13-D (full-fit):")
for class_id in range(NUM_CLASSES):
    print(f"  {CLASS_NAMES[class_id]:<18s}: {active_models_per_class[class_id]} model(s)")

max_reasonable_gap = 0.03  # QWK points; a documented, stated threshold, not tuned post-hoc.
unstable_variants = stability_df.loc[stability_df["optimism_gap"] > max_reasonable_gap, "variant"].tolist()
if unstable_variants:
    print(f"\nWARNING: variant(s) {unstable_variants} show in-sample QWK more than {max_reasonable_gap} "
          "above their out-of-fold QWK - a sign of overfitting to validation noise. These should be "
          "viewed with caution in Section 19's selection, regardless of their in-sample numbers.")
else:
    print(f"\nNo variant shows an optimism gap above {max_reasonable_gap} QWK - no strong overfitting signal.")

## 19. Select final ensemble variant

Selection uses **only the out-of-fold numbers** from Section 17: highest QWK, then highest
macro-F1, then lowest NLL, then lowest ECE, then prefer the simpler ensemble (A simplest,
D most complex) as the final tie-break. The class-specialist variant (D) is **not**
automatically preferred — it must actually win this rule to be selected.

In [ ]:
variant_complexity_rank = {"A": 0, "B": 1, "C": 2, "D": 3}  # lower = simpler = preferred on ties

selection_candidates = []
for variant in ["A", "B", "C", "D"]:
    m = oof_variant_metrics[variant]
    selection_candidates.append((
        -m["qwk"], -m["macro_f1"], m["nll"], m["ece"], variant_complexity_rank[variant], variant
    ))
selection_candidates.sort()
FINAL_VARIANT = selection_candidates[0][-1]

print("Selection ranking (best first):")
for rank in selection_candidates:
    print(f"  variant {rank[-1]}: qwk={-rank[0]:.4f} macro_f1={-rank[1]:.4f} nll={rank[2]:.4f} ece={rank[3]:.4f}")

print(f"\nFINAL SELECTED VARIANT: M13-{FINAL_VARIANT}")
print(f"OOF QWK={oof_variant_metrics[FINAL_VARIANT]['qwk']:.4f}  "
      f"macro-F1={oof_variant_metrics[FINAL_VARIANT]['macro_f1']:.4f}  "
      f"NLL={oof_variant_metrics[FINAL_VARIANT]['nll']:.4f}  "
      f"ECE={oof_variant_metrics[FINAL_VARIANT]['ece']:.4f}")

## 20. Refit calibration on full validation set

The OOF protocol (Section 17) exists to *select* the variant honestly — once selected, the
final deployed system's temperatures and weights are refit on **all** validation rows (more
data than any 4-fold subset) for the actual frozen system. `fitted_temperatures`,
`full_fit_global_weights`, and `full_fit_class_weights` were already computed in Section 18;
reused here as the final values.

In [ ]:
final_temperatures = fitted_temperatures
final_global_weights = full_fit_global_weights
final_class_weights = full_fit_class_weights

final_probs = in_sample_probs[FINAL_VARIANT]
final_preds = final_probs.argmax(axis=1)

print(f"Refit complete for the selected variant: M13-{FINAL_VARIANT}")
print("Final per-model temperatures:", {name: round(t, 4) for name, t in final_temperatures.items()})
if FINAL_VARIANT == "C":
    print("Final global weights:", dict(zip(CANDIDATE_MODEL_NAMES, np.round(final_global_weights, 4))))

## 21. Freeze final model list and weights

**These parameters are frozen after validation. They must not be changed after viewing
internal-test results.**

In [ ]:
print("=" * 70)
print("  THESE PARAMETERS ARE FROZEN AFTER VALIDATION.")
print("  THEY MUST NOT BE CHANGED AFTER VIEWING INTERNAL-TEST RESULTS.")
print("=" * 70)

final_ensemble_config = {
    "selected_variant": f"M13-{FINAL_VARIANT}",
    "selection_rule": "oof_qwk_then_macro_f1_then_nll_then_ece_then_simplicity",
    "models_included": CANDIDATE_MODEL_NAMES,
    "logit_sources": {name: loaded_model_data[name]["logit_source"] for name in CANDIDATE_MODEL_NAMES},
    "n_validation_rows": len(aligned_ids),
    "oof_metrics_at_selection": oof_variant_metrics[FINAL_VARIANT],
    "fold_seed": FOLD_SEED,
    "n_oof_folds": N_OOF_FOLDS,
    "frozen_warning": (
        "These parameters are frozen after validation. They must not be changed after "
        "viewing internal-test results."
    ),
}
with open(LOG_DIR / "final_ensemble_config.json", "w") as f:
    json.dump(final_ensemble_config, f, indent=2, default=float)
print(f"Saved -> {LOG_DIR / 'final_ensemble_config.json'}")

with open(LOG_DIR / "model_temperatures.json", "w") as f:
    json.dump(final_temperatures, f, indent=2)
print(f"Saved -> {LOG_DIR / 'model_temperatures.json'}")

global_weights_dict = dict(zip(CANDIDATE_MODEL_NAMES, final_global_weights.tolist()))
with open(LOG_DIR / "global_ensemble_weights.json", "w") as f:
    json.dump(global_weights_dict, f, indent=2)
print(f"Saved -> {LOG_DIR / 'global_ensemble_weights.json'}")

class_specialist_weights_df = pd.DataFrame(
    final_class_weights, index=CANDIDATE_MODEL_NAMES, columns=CLASS_NAMES,
)
class_specialist_weights_df.to_csv(LOG_DIR / "class_specialist_weights.csv")
print(f"Saved -> {LOG_DIR / 'class_specialist_weights.csv'}")

oof_predictions_df = pd.DataFrame({
    "image_id": aligned_ids, "true_label": aligned_labels,
    **{f"oof_prob_{c}": oof_probs[FINAL_VARIANT][:, c] for c in range(NUM_CLASSES)},
    "oof_predicted_label": oof_probs[FINAL_VARIANT].argmax(axis=1),
})
oof_predictions_df.to_csv(LOG_DIR / "validation_oof_predictions.csv", index=False)
print(f"Saved -> {LOG_DIR / 'validation_oof_predictions.csv'}")

fullfit_predictions_df = pd.DataFrame({
    "image_id": aligned_ids, "true_label": aligned_labels,
    **{f"prob_{c}": final_probs[:, c] for c in range(NUM_CLASSES)},
    "predicted_label": final_preds,
})
fullfit_predictions_df.to_csv(LOG_DIR / "validation_fullfit_predictions.csv", index=False)
print(f"Saved -> {LOG_DIR / 'validation_fullfit_predictions.csv'}")

np.save(LOG_DIR / "validation_final_probabilities.npy", final_probs)
np.save(LOG_DIR / "validation_final_labels.npy", aligned_labels)
pd.DataFrame({"image_id": aligned_ids}).to_csv(LOG_DIR / "validation_final_image_ids.csv", index=False)
print(f"Saved -> {LOG_DIR / 'validation_final_probabilities.npy'}")
print(f"Saved -> {LOG_DIR / 'validation_final_labels.npy'}")
print(f"Saved -> {LOG_DIR / 'validation_final_image_ids.csv'}")

## 22. Full-validation descriptive metrics

Descriptive only (the OOF numbers in Section 17 are the honest ones): full metric suite for
the final frozen system on the complete validation set, plus per-class precision/recall/F1
and the specific error transitions requested.

In [ ]:
final_core = compute_core_metrics(aligned_labels, final_preds)
final_nll = negative_log_likelihood(aligned_labels, final_probs)
final_ece = expected_calibration_error(aligned_labels, final_probs)["ece"]
final_brier = brier_score(aligned_labels, final_probs)

per_class_precision = precision_score(aligned_labels, final_preds, average=None, labels=list(range(NUM_CLASSES)), zero_division=0)
per_class_recall = recall_score(aligned_labels, final_preds, average=None, labels=list(range(NUM_CLASSES)), zero_division=0)
per_class_f1 = f1_score(aligned_labels, final_preds, average=None, labels=list(range(NUM_CLASSES)), zero_division=0)
per_class_support = np.bincount(aligned_labels, minlength=NUM_CLASSES)

per_class_metrics_df = pd.DataFrame({
    "class": CLASS_NAMES, "precision": per_class_precision, "recall": per_class_recall,
    "f1": per_class_f1, "support": per_class_support,
})
per_class_metrics_df.to_csv(LOG_DIR / "per_class_metrics.csv", index=False)
print(per_class_metrics_df.to_string(index=False))
print(f"Saved -> {LOG_DIR / 'per_class_metrics.csv'}")

mild_to_no_dr = int(np.sum((aligned_labels == 1) & (final_preds == 0)))
moderate_to_no_dr = int(np.sum((aligned_labels == 2) & (final_preds == 0)))
severe_to_moderate = int(np.sum((aligned_labels == 3) & (final_preds == 2)))
pdr_to_severe = int(np.sum((aligned_labels == 4) & (final_preds == 3)))
large_errors = int(np.sum(np.abs(final_preds - aligned_labels) >= 2))
high_conf_wrong = int(np.sum((final_probs.max(axis=1) >= HIGH_CONFIDENCE_THRESHOLD) & (final_preds != aligned_labels)))

full_validation_ensemble_metrics = {
    **final_core, "nll": final_nll, "ece": final_ece, "brier": final_brier,
    "mild_to_no_dr": mild_to_no_dr, "moderate_to_no_dr": moderate_to_no_dr,
    "severe_to_moderate": severe_to_moderate, "pdr_to_severe": pdr_to_severe,
    "large_errors_ge2": large_errors, "high_confidence_wrong": high_conf_wrong,
}
pd.DataFrame([full_validation_ensemble_metrics]).to_csv(LOG_DIR / "full_validation_ensemble_metrics.csv", index=False)
print()
for key, value in full_validation_ensemble_metrics.items():
    print(f"  {key:<22s}: {value}")
print(f"\nSaved -> {LOG_DIR / 'full_validation_ensemble_metrics.csv'}")

## 23. Reliability diagrams

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
for ax, probs, title in [
    (axes[0], ensemble_a_uncalibrated_equal_average(aligned_logits, CANDIDATE_MODEL_NAMES), "Before calibration (M13-A)"),
    (axes[1], final_probs, f"After calibration (selected M13-{FINAL_VARIANT})"),
]:
    ece_result = expected_calibration_error(aligned_labels, probs)
    bins_df = ece_result["bins"].dropna()
    ax.bar(bins_df["bin_lo"], bins_df["accuracy"], width=1.0 / 15, align="edge", alpha=0.7, label="Observed accuracy", color="#3498DB")
    ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
    ax.set_xlabel("Confidence"); ax.set_ylabel("Accuracy")
    ax.set_title(f"{title}\nECE={ece_result['ece']:.4f}", fontweight="bold")
    ax.legend(); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "reliability_diagram_before_after.png", dpi=140, bbox_inches="tight")
plt.show()
print(f"Saved -> {FIGURE_DIR / 'reliability_diagram_before_after.png'}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
temps = [final_temperatures[name] for name in CANDIDATE_MODEL_NAMES]
ax.bar(CANDIDATE_MODEL_NAMES, temps, color="#8E44AD")
ax.axhline(1.0, color="black", linestyle="--", label="T=1 (no change)")
ax.set_ylabel("Fitted temperature")
ax.set_title("Per-model fitted temperature (full-validation refit)", fontweight="bold")
ax.legend()
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "model_temperature_comparison.png", dpi=140, bbox_inches="tight")
plt.show()
print(f"Saved -> {FIGURE_DIR / 'model_temperature_comparison.png'}")

In [ ]:
metric_names = ["NLL", "ECE", "Brier"]
before_after = {
    "Before": [
        negative_log_likelihood(aligned_labels, ensemble_a_uncalibrated_equal_average(aligned_logits, CANDIDATE_MODEL_NAMES)),
        expected_calibration_error(aligned_labels, ensemble_a_uncalibrated_equal_average(aligned_logits, CANDIDATE_MODEL_NAMES))["ece"],
        brier_score(aligned_labels, ensemble_a_uncalibrated_equal_average(aligned_logits, CANDIDATE_MODEL_NAMES)),
    ],
    "After (selected)": [final_nll, final_ece, final_brier],
}
nll_ece_brier_df = pd.DataFrame(before_after, index=metric_names)

fig, ax = plt.subplots(figsize=(7, 5))
nll_ece_brier_df.plot(kind="bar", ax=ax, color=["#E74C3C", "#2ECC71"])
ax.set_title("NLL / ECE / Brier before vs after calibration", fontweight="bold")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "nll_ece_brier_comparison.png", dpi=140, bbox_inches="tight")
plt.show()
print(f"Saved -> {FIGURE_DIR / 'nll_ece_brier_comparison.png'}")

In [ ]:
final_confidences = final_probs.max(axis=1)
correct_mask = final_preds == aligned_labels

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(final_confidences[correct_mask], bins=15, alpha=0.6, label="Correct", color="#2ECC71")
ax.hist(final_confidences[~correct_mask], bins=15, alpha=0.6, label="Wrong", color="#E74C3C")
ax.axvline(HIGH_CONFIDENCE_THRESHOLD, color="black", linestyle="--", label=f"High-confidence threshold ({HIGH_CONFIDENCE_THRESHOLD})")
ax.set_xlabel("Confidence"); ax.set_ylabel("Count")
ax.set_title(f"Confidence distribution: correct vs wrong (M13-{FINAL_VARIANT})", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR / "confidence_histogram_correct_vs_wrong.png", dpi=140, bbox_inches="tight")
plt.show()
print(f"Saved -> {FIGURE_DIR / 'confidence_histogram_correct_vs_wrong.png'}")

## 24. Per-class specialist analysis

For every model and class: precision, recall, F1, one-vs-rest average precision, and a
reliability check across out-of-fold splits — a specialist is only trusted if its advantage
is stable, not a one-off high F1 on a small class.

In [ ]:
specialist_analysis_rows = []
for model_name in CANDIDATE_MODEL_NAMES:
    probs = apply_temperature(aligned_logits[model_name], final_temperatures[model_name])
    preds = probs.argmax(axis=1)
    for class_id in range(NUM_CLASSES):
        class_precision = precision_score(aligned_labels == class_id, preds == class_id, zero_division=0)
        class_recall = recall_score(aligned_labels == class_id, preds == class_id, zero_division=0)
        class_f1 = f1_score(aligned_labels == class_id, preds == class_id, zero_division=0)
        class_ap = average_precision_score((aligned_labels == class_id).astype(int), probs[:, class_id])
        n_correct = int(np.sum((aligned_labels == class_id) & (preds == class_id)))

        wrong_mask = (aligned_labels == class_id) & (preds != class_id)
        if wrong_mask.sum() > 0:
            destinations, counts = np.unique(preds[wrong_mask], return_counts=True)
            common_destination = CLASS_NAMES[int(destinations[np.argmax(counts)])]
        else:
            common_destination = "n/a"

        specialist_analysis_rows.append({
            "model": model_name, "class": CLASS_NAMES[class_id],
            "precision": class_precision, "recall": class_recall, "f1": class_f1,
            "average_precision": class_ap, "n_correct": n_correct,
            "common_confusion_destination": common_destination,
        })

specialist_analysis_df = pd.DataFrame(specialist_analysis_rows)
print(specialist_analysis_df.round(4).to_string(index=False))

specialist_analysis_df.to_csv(LOG_DIR / "class_specialist_analysis.csv", index=False)
print(f"Saved -> {LOG_DIR / 'class_specialist_analysis.csv'}")

In [ ]:
# Reliability check: does each candidate specialist's per-class F1 advantage hold up across
# out-of-fold splits, not just on the full-fit numbers above?
oof_specialist_f1 = {name: {c: [] for c in range(NUM_CLASSES)} for name in CANDIDATE_MODEL_NAMES}
for fold_idx in range(N_OOF_FOLDS):
    held_out_mask = fold_assignment == fold_idx
    for model_name in CANDIDATE_MODEL_NAMES:
        fold_temp = fit_temperature(aligned_logits[model_name][fold_assignment != fold_idx], aligned_labels[fold_assignment != fold_idx])
        fold_probs = apply_temperature(aligned_logits[model_name][held_out_mask], fold_temp)
        fold_preds = fold_probs.argmax(axis=1)
        fold_labels = aligned_labels[held_out_mask]
        for class_id in range(NUM_CLASSES):
            oof_specialist_f1[model_name][class_id].append(
                f1_score(fold_labels == class_id, fold_preds == class_id, zero_division=0)
            )

selected_specialists = {}
for class_id in range(NUM_CLASSES):
    class_rows = specialist_analysis_df[specialist_analysis_df["class"] == CLASS_NAMES[class_id]]
    ranked = class_rows.sort_values("f1", ascending=False)
    best_model = ranked.iloc[0]["model"]
    best_f1 = ranked.iloc[0]["f1"]
    oof_f1_values = oof_specialist_f1[best_model][class_id]
    oof_f1_std = float(np.std(oof_f1_values))
    stable = oof_f1_std < 0.10  # a stated, documented stability threshold
    selected_specialists[CLASS_NAMES[class_id]] = {
        "model": best_model, "full_fit_f1": float(best_f1),
        "oof_f1_mean": float(np.mean(oof_f1_values)), "oof_f1_std": oof_f1_std, "stable": stable,
    }

print("Selected class specialists (full-fit best F1, with OOF stability check):")
for class_name, info in selected_specialists.items():
    stability_note = "STABLE" if info["stable"] else "UNSTABLE - use with caution"
    print(f"  {class_name:<18s}: {info['model']:<16s} F1={info['full_fit_f1']:.4f}  "
          f"OOF mean={info['oof_f1_mean']:.4f} +/- {info['oof_f1_std']:.4f}  [{stability_note}]")

## 25. Error complementarity

For every pair of candidate models: agreement/disagreement breakdown, Jaccard similarity of
error sets, disagreement rate, and the oracle upper bound (accuracy if a hypothetical oracle
always picked whichever model was correct) — this demonstrates whether real ensemble
diversity exists, or whether the models are simply making the same mistakes.

In [ ]:
model_preds_calibrated = {
    name: apply_temperature(aligned_logits[name], final_temperatures[name]).argmax(axis=1)
    for name in CANDIDATE_MODEL_NAMES
}
model_correct = {name: (model_preds_calibrated[name] == aligned_labels) for name in CANDIDATE_MODEL_NAMES}
model_error_sets = {name: set(np.array(aligned_ids)[~model_correct[name]]) for name in CANDIDATE_MODEL_NAMES}

complementarity_rows = []
for model_a, model_b in combinations(CANDIDATE_MODEL_NAMES, 2):
    both_correct = int(np.sum(model_correct[model_a] & model_correct[model_b]))
    both_wrong = int(np.sum(~model_correct[model_a] & ~model_correct[model_b]))
    a_correct_b_wrong = int(np.sum(model_correct[model_a] & ~model_correct[model_b]))
    a_wrong_b_correct = int(np.sum(~model_correct[model_a] & model_correct[model_b]))
    disagreement_rate = float(np.mean(model_preds_calibrated[model_a] != model_preds_calibrated[model_b]))
    error_a, error_b = model_error_sets[model_a], model_error_sets[model_b]
    union = error_a | error_b
    jaccard_similarity = len(error_a & error_b) / len(union) if len(union) > 0 else float("nan")

    complementarity_rows.append({
        "model_a": model_a, "model_b": model_b,
        "both_correct": both_correct, "both_wrong": both_wrong,
        "a_correct_b_wrong": a_correct_b_wrong, "a_wrong_b_correct": a_wrong_b_correct,
        "jaccard_error_similarity": jaccard_similarity, "disagreement_rate": disagreement_rate,
    })

complementarity_df = pd.DataFrame(complementarity_rows)
print(complementarity_df.round(4).to_string(index=False))
complementarity_df.to_csv(LOG_DIR / "model_error_complementarity.csv", index=False)
print(f"\nSaved -> {LOG_DIR / 'model_error_complementarity.csv'}")

any_correct = np.zeros(len(aligned_labels), dtype=bool)
for name in CANDIDATE_MODEL_NAMES:
    any_correct |= model_correct[name]
oracle_accuracy = float(any_correct.mean())
best_single_qwk = max(single_model_uncalibrated_df["qwk"])
print(f"\nOracle upper-bound accuracy (any model correct): {oracle_accuracy:.4f}")
print(f"Best single-model uncalibrated QWK              : {best_single_qwk:.4f}")
print(f"Selected M13-{FINAL_VARIANT} OOF QWK                        : {oof_variant_metrics[FINAL_VARIANT]['qwk']:.4f}")

## 26. M12-to-M13 error transitions

What the selected M13 ensemble does relative to M12 specifically, since M12 is the strongest
single MaxViT model and the most direct "did the ensemble help" comparison.

In [ ]:
m12_correct = model_correct.get("M12")
if m12_correct is None:
    print("M12 not present among aligned models; skipping M12-to-M13 transition analysis.")
    ensemble_error_transitions = pd.DataFrame()
else:
    m13_correct = final_preds == aligned_labels

    corrects_from_m12 = int(np.sum(~m12_correct & m13_correct))
    wrong_despite_m12_correct = int(np.sum(m12_correct & ~m13_correct))
    continues_to_misclassify = int(np.sum(~m12_correct & ~m13_correct))

    m12_preds = apply_temperature(aligned_logits["M12"], final_temperatures["M12"]).argmax(axis=1)
    changes_mild_to_no_dr = int(np.sum(
        (aligned_labels == 1) & (m12_preds != 0) & (final_preds == 0)
    ))
    changes_severe_to_moderate = int(np.sum(
        (aligned_labels == 3) & (m12_preds != 2) & (final_preds == 2)
    ))

    print(f"Corrected from M12 (M12 wrong, M13 correct)        : {corrects_from_m12}")
    print(f"Made wrong despite M12 correct (M12 right, M13 wrong): {wrong_despite_m12_correct}")
    print(f"Continues to misclassify (both wrong)               : {continues_to_misclassify}")
    print(f"New Mild -> No DR errors introduced by M13          : {changes_mild_to_no_dr}")
    print(f"New Severe -> Moderate errors introduced by M13     : {changes_severe_to_moderate}")

    ensemble_error_transitions = pd.DataFrame([{
        "corrected_from_m12": corrects_from_m12,
        "wrong_despite_m12_correct": wrong_despite_m12_correct,
        "continues_to_misclassify": continues_to_misclassify,
        "new_mild_to_no_dr_errors": changes_mild_to_no_dr,
        "new_severe_to_moderate_errors": changes_severe_to_moderate,
    }])
    ensemble_error_transitions.to_csv(LOG_DIR / "ensemble_error_transitions.csv", index=False)
    print(f"\nSaved -> {LOG_DIR / 'ensemble_error_transitions.csv'}")

## 27. High-confidence error analysis

Confidence ≥ 0.80. Calibration should ideally reduce unjustified confidence even where class
predictions are unchanged.

In [ ]:
high_confidence_rows = []
for name in CANDIDATE_MODEL_NAMES + [f"M13-{FINAL_VARIANT} (ensemble)"]:
    if name.startswith("M13-"):
        probs, preds = final_probs, final_preds
    else:
        probs = apply_temperature(aligned_logits[name], final_temperatures[name])
        preds = probs.argmax(axis=1)

    confidences = probs.max(axis=1)
    wrong_mask = preds != aligned_labels
    high_conf_wrong_mask = wrong_mask & (confidences >= HIGH_CONFIDENCE_THRESHOLD)

    n_high_conf_errors = int(high_conf_wrong_mask.sum())
    mean_conf_wrong = float(confidences[wrong_mask].mean()) if wrong_mask.sum() > 0 else float("nan")
    max_conf_wrong = float(confidences[wrong_mask].max()) if wrong_mask.sum() > 0 else float("nan")

    high_confidence_rows.append({
        "model": name, "n_high_confidence_errors": n_high_conf_errors,
        "mean_confidence_on_wrong": mean_conf_wrong, "max_confidence_on_wrong": max_conf_wrong,
    })

high_confidence_df = pd.DataFrame(high_confidence_rows)
print(high_confidence_df.round(4).to_string(index=False))

high_confidence_df.to_csv(LOG_DIR / "high_confidence_error_summary.csv", index=False)
print(f"Saved -> {LOG_DIR / 'high_confidence_error_summary.csv'}")

## 28. Save all outputs

Remaining required figures and `config.json`.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
plot_metrics = ["qwk", "macro_f1", "balanced_accuracy"]
model_rows = single_model_uncalibrated_df[plot_metrics].copy()
model_rows.loc[f"M13-{FINAL_VARIANT} (ensemble)"] = [final_core[m] for m in plot_metrics]
model_rows.plot(kind="bar", ax=ax)
ax.set_title("Single models vs selected M13 ensemble", fontweight="bold")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "single_model_vs_ensemble_metrics.png", dpi=140, bbox_inches="tight")
plt.show()
print(f"Saved -> {FIGURE_DIR / 'single_model_vs_ensemble_metrics.png'}")

In [ ]:
class_f1_rows = {name: per_class_f1 if name.startswith("M13") else
                  f1_score(aligned_labels, apply_temperature(aligned_logits[name], final_temperatures[name]).argmax(axis=1),
                           average=None, labels=list(range(NUM_CLASSES)), zero_division=0)
                  for name in CANDIDATE_MODEL_NAMES + [f"M13-{FINAL_VARIANT}"]}
class_f1_comparison_df = pd.DataFrame(class_f1_rows, index=CLASS_NAMES)

fig, ax = plt.subplots(figsize=(10, 5.5))
class_f1_comparison_df.plot(kind="bar", ax=ax)
ax.set_ylabel("F1 score")
ax.set_title("Per-class F1: single models vs selected M13 ensemble", fontweight="bold")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "class_f1_ensemble_comparison.png", dpi=140, bbox_inches="tight")
plt.show()
print(f"Saved -> {FIGURE_DIR / 'class_f1_ensemble_comparison.png'}")

In [ ]:
cm_counts = confusion_matrix(aligned_labels, final_preds)
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(cm_counts, cmap="Purples")
ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right"); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"Selected M13-{FINAL_VARIANT} ensemble -- confusion matrix", fontweight="bold")
thresh = cm_counts.max() / 2
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, cm_counts[i, j], ha="center", va="center", color="white" if cm_counts[i, j] > thresh else "black")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "ensemble_confusion_matrix.png", dpi=140, bbox_inches="tight")
plt.show()
print(f"Saved -> {FIGURE_DIR / 'ensemble_confusion_matrix.png'}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
im = ax.imshow(final_class_weights, cmap="YlOrRd", aspect="auto", vmin=0, vmax=1)
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASS_NAMES, rotation=30, ha="right")
ax.set_yticks(range(len(CANDIDATE_MODEL_NAMES))); ax.set_yticklabels(CANDIDATE_MODEL_NAMES)
for i in range(len(CANDIDATE_MODEL_NAMES)):
    for j in range(NUM_CLASSES):
        ax.text(j, i, f"{final_class_weights[i, j]:.2f}", ha="center", va="center", fontsize=8)
ax.set_title("M13-D class-specialist weights (models x classes)", fontweight="bold")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "class_specialist_weights_heatmap.png", dpi=140, bbox_inches="tight")
plt.show()
print(f"Saved -> {FIGURE_DIR / 'class_specialist_weights_heatmap.png'}")

In [ ]:
error_transition_labels = ["Mild->No DR", "Moderate->No DR", "Severe->Moderate", "PDR->Severe"]
error_transition_rows = {}
for name in CANDIDATE_MODEL_NAMES + [f"M13-{FINAL_VARIANT}"]:
    preds = final_preds if name.startswith("M13") else apply_temperature(aligned_logits[name], final_temperatures[name]).argmax(axis=1)
    error_transition_rows[name] = [
        int(np.sum((aligned_labels == 1) & (preds == 0))),
        int(np.sum((aligned_labels == 2) & (preds == 0))),
        int(np.sum((aligned_labels == 3) & (preds == 2))),
        int(np.sum((aligned_labels == 4) & (preds == 3))),
    ]
error_transition_df = pd.DataFrame(error_transition_rows, index=error_transition_labels)

fig, ax = plt.subplots(figsize=(10, 5.5))
error_transition_df.T.plot(kind="bar", ax=ax)
ax.set_ylabel("Count")
ax.set_title("Targeted error-transition counts: single models vs selected M13 ensemble", fontweight="bold")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "error_transition_comparison.png", dpi=140, bbox_inches="tight")
plt.show()
print(f"Saved -> {FIGURE_DIR / 'error_transition_comparison.png'}")

In [ ]:
m13_config = {
    "experiment_id": EXPERIMENT_ID,
    "run_name": RUN_NAME,
    "trains_new_network": False,
    "internal_test_loaded": False,
    "candidate_models": CANDIDATE_MODEL_NAMES,
    "logit_sources": {name: loaded_model_data[name]["logit_source"] for name in CANDIDATE_MODEL_NAMES},
    "n_validation_rows": len(aligned_ids),
    "fold_seed": FOLD_SEED,
    "n_oof_folds": N_OOF_FOLDS,
    "high_confidence_threshold": HIGH_CONFIDENCE_THRESHOLD,
    "weight_grid": WEIGHT_GRID,
    "selected_variant": f"M13-{FINAL_VARIANT}",
    "selection_rule": "oof_qwk_then_macro_f1_then_nll_then_ece_then_simplicity",
}
with open(LOG_DIR / "config.json", "w") as f:
    json.dump(m13_config, f, indent=2)
print(f"Saved -> {LOG_DIR / 'config.json'}")

## 29. Limitations

- **Validation data performs both development and final parameter fitting** — even with the
  out-of-fold protocol, every design decision in this notebook (which variant to try, which
  models to include, the weight-grid resolution, the specialist cap) was made by a human
  looking at validation results, which is a softer, harder-to-eliminate form of the same
  optimism the OOF protocol targets at the parameter-fitting level specifically.
- **Out-of-fold analysis reduces but does not eliminate selection optimism** — the final
  variant was *chosen* by looking at OOF numbers across all four candidates; the reported OOF
  metric for the winning variant is still, by construction, the best of four correlated
  observations, not an independent held-out estimate of that variant alone.
- **Specialist choices may be unstable for Severe and PDR** because they contain fewer
  examples than No DR/Mild/Moderate — the OOF-stability check in Section 24 flags this
  explicitly rather than presenting every specialist choice with equal confidence.
- **Calibration improves probability reliability but does not guarantee better class
  predictions** — as demonstrated by the assertion in Section 12, QWK/accuracy/F1 are
  mathematically unchanged by temperature scaling of a single model in isolation.
- **The internal test set remains completely untouched** and is required for the final,
  genuinely unbiased comparison — nothing in this notebook substitutes for that.

## 30. Experiment conclusion

To be completed after execution:

- Did calibration reduce NLL, ECE and Brier score?
- Did equal averaging improve over M12?
- Did weighting improve over equal averaging?
- Did class specialists improve Mild or Severe recognition?
- Did specialist weighting hurt QWK or another class?
- Were improvements present in out-of-fold results or only full-validation fitting?
- Did M13 reduce high-confidence errors?
- Which model disagreements provided useful diversity?
- Is M13 preferable to M12?
- Which frozen system should proceed to A01?